In [1]:
%load_ext autoreload
%autoreload 2
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness

/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


# Interference Results

## Get IF data

In [47]:
app_name = "deepspeed-dlio-step100" 
cp_interference = "/p/lustre3/pandey2/logs/Results_Checkpoint"

data_events_obj = DFGrepInterferencePartitionBased(app_name=app_name, operation="data", cp_dir = cp_dir, existing=True)
data = data_events_obj.inter.compute()


data["Operation"] = "Data"
data["Interfered"] = np.where(data['deg_caller'] < 1 / data['interference'], "interfered", "not-interfered")
data['valid'] = (data['deg_caller'] > 1) & (data['dur'] > 0) & (data['interference'] > 0)



## Get Original Data

In [4]:


if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage-pegasus-2mass-2deg-4node" #cosmoflow cm1
# app_name = "1000_genome_pegasus_node_16"
# app_name = "cm1"
# app_name = "deepspeed-dlio-step100"
# app_name = "deepspeed-dlio-scr-step100"
# app_name = "bert"
# app_name = "unet3d"
# app_name = "resnet50"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint"

condition_fn = None #

if app_name == "cm1":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cm1/APP/node-32/v1/COMPACT/*.pfw.gz"
    # cp_dir = "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-step100":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-scr-step100":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "montage-mpi-2mass-7deg":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/mpi-2mass-7deg/node-16/v1/RAW/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name =="montage-pegasus-2mass-2deg-4node":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/RAW/*.pfw.gz"
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage-pegasus-2mass-2deg-4node/RAW/*.pfw.gz" # Raw Copied
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "resnet50":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/resnet50/dlio-v100/node-4/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "unet3d":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/unet3d/dlio-v100/node-16/v2/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "bert":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/bert/v100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "1000_genome_pegasus_node_16":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/1000-genome/pegasus/node-16/v3/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"
    
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [10:17:26] Initialized Client with 96 workers and link http://134.9.71.28:41665/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


[INFO] [10:17:31] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


In [5]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))
def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    return d

load_cols = {'size': "int64[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}

analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)
if app_name in ["deepspeed-dlio-step100","deepspeed-dlio-scr-step100","resnet50","unet3d"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']]
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name == "montage-pegasus-2mass-2deg-4node":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["1000_genome_pegasus_node_16"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]


elif app_name == "bert":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["cm1"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]




[INFO] [10:17:46] Created index for 30449 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [10:17:46] Total size of all files are <dask.bag.core.Item object at 0x1554acb54be0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [10:17:46] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [10:17:52] Loading 30660 batches out of 30449 files and has 75603259 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [10:29:32] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [10:29:32] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [10]:
analyzer.events.name.unique().compute()

0         start
1       __xstat
2         fopen
3        fclose
4       opendir
5          open
6          read
7         close
8        access
9          fork
10         dup2
11        fread
12       unlink
13          end
14      fopen64
15    __xstat64
16        fseek
17       remove
18       fwrite
19        mkdir
20    ftruncate
21         mmap
22     readlink
23     __fxstat
24        write
25        lseek
26        fcntl
27        rmdir
Name: name, dtype: string

In [5]:
analyze_df.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,id
0,__xstat,POSIX,<NA>,81028810,81028827,17,81,/usr/WS2,corona201,0
1,fopen,STDIO,<NA>,81029009,81029017,8,81,/proc,corona201,1
2,fclose,STDIO,<NA>,81029046,81029048,2,81,/proc,corona201,2
3,fopen,STDIO,<NA>,81029058,81029076,18,81,/proc,corona201,3
4,fclose,STDIO,<NA>,81032110,81032113,3,81,/proc,corona201,4


In [5]:
result_df1["mount_point"] = result_df1["mount_point"].where(result_df1["mount_point"].str.startswith("/"), "/p/lustre3")

In [71]:
result_df1.query('size > 0').head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,compute_time,io_time,app_io_time,total_time,fhash,phase,size,value,mount_point,hostname
10417,open64,POSIX,0,2837486,2837486,16065659427268196451,26612969,26614101,1132,<NA>,...,<NA>,1132,<NA>,1132,1.6893228754985929e+19,2,31,<NA>,/usr/WS2,corona202
10420,read,POSIX,0,2837486,2837486,16065659427268196451,26614156,26614372,216,<NA>,...,<NA>,216,<NA>,216,1.6893228754985929e+19,2,3130,<NA>,/usr/WS2,corona202
10436,open64,POSIX,0,2837486,2837486,16065659427268196451,26639480,26640110,630,<NA>,...,<NA>,630,<NA>,630,1.577627230041033e+19,2,31,<NA>,/usr/WS2,corona202
10439,read,POSIX,0,2837486,2837486,16065659427268196451,26640147,26640304,157,<NA>,...,<NA>,157,<NA>,157,1.577627230041033e+19,2,1272,<NA>,/usr/WS2,corona202
10471,open64,POSIX,0,2837486,2837486,16065659427268196451,26689502,26690053,551,<NA>,...,<NA>,551,<NA>,551,1.4436710900643502e+18,2,31,<NA>,/usr/WS2,corona202


In [60]:
cols = ["cat", "size", "ts", "te", "mount_point", "hostname"]
filtered = data.query("hostname=='corona222' and Interfered == 'interfered' and valid == 'True'")


In [57]:
filtered

,name,cat,size,size_category,ts,te,dur,trange,mount_point,hostname,deg_caller,deg_other,min_dur,interference,Operation,Interfered,valid
199,read,POSIX,4025,2^1 KiB - 2^2 KiB,106512,107228,716,0,/collab/usr/gapps,corona222,16,1.0,3.0,0.00419,Data,interfered,True
200,read,POSIX,4025,2^1 KiB - 2^2 KiB,106520,107229,709,0,/collab/usr/gapps,corona222,16,1.0,3.0,0.004231,Data,interfered,True
201,read,POSIX,4025,2^1 KiB - 2^2 KiB,106531,107230,699,0,/collab/usr/gapps,corona222,16,1.0,3.0,0.004292,Data,interfered,True
202,read,POSIX,4025,2^1 KiB - 2^2 KiB,106573,107233,660,0,/collab/usr/gapps,corona222,16,1.0,3.0,0.004545,Data,interfered,True
203,read,POSIX,4025,2^1 KiB - 2^2 KiB,106578,107234,656,0,/collab/usr/gapps,corona222,16,1.0,3.0,0.004573,Data,interfered,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9192,read,POSIX,2,2^0 - 2^1 B,8665518638,8665518647,9,288,/sys,corona222,2,1.0,4.0,0.444444,Data,interfered,True
9193,read,POSIX,2,2^0 - 2^1 B,8665518650,8665518662,12,288,/sys,corona222,2,1.0,4.0,0.333333,Data,interfered,True
9414,read,POSIX,14,2^3 - 2^4 B,8665527118,8665527126,8,288,/sys,corona222,2,1.0,3.0,0.375,Data,interfered,True
10878,write,POSIX,65378,2^5 KiB - 2^6 KiB,8665337169,8665337230,61,288,/usr/workspace,corona222,2,1.0,28.0,0.459016,Data,interfered,True


In [58]:
cols

['cat', 'size', 'ts', 'te', 'dur', 'trange', 'mount_point', 'hostname']

In [59]:
result_df1.head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,compute_time,io_time,app_io_time,total_time,fhash,phase,size,value,mount_point,hostname
0,__lxstat64,POSIX,0,2837486,2837486,16065659427268196451,26180227,26180242,15,<NA>,...,<NA>,15,<NA>,15,9.018683717544912e+18,2,<NA>,<NA>,/usr/WS2,corona202
1,__lxstat64,POSIX,0,2837486,2837486,16065659427268196451,26180252,26180269,17,<NA>,...,<NA>,17,<NA>,17,6.0707729057472e+18,2,<NA>,<NA>,/usr/WS2,corona202
2,__xstat64,POSIX,0,2837486,2837486,16065659427268196451,26180292,26180654,362,<NA>,...,<NA>,362,<NA>,362,1.1650986456545004e+19,2,<NA>,<NA>,/usr/WS2,corona202
3,__lxstat64,POSIX,0,2837486,2837486,16065659427268196451,26180680,26180683,3,<NA>,...,<NA>,3,<NA>,3,8.709334012377346e+18,2,<NA>,<NA>,/usr,corona202
4,__lxstat64,POSIX,0,2837486,2837486,16065659427268196451,26180694,26180697,3,<NA>,...,<NA>,3,<NA>,3,1.194672913741415e+19,2,<NA>,<NA>,/usr/WS2,corona202


In [61]:
new_df = result_df1.merge(filtered[cols], on= cols, how="inner")


In [69]:
new_df.compute()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,compute_time,io_time,app_io_time,total_time,fhash,phase,size,value,mount_point,hostname
0,read,POSIX,0,2033896,2033896,7096598225222401591,12122032,12122755,723,<NA>,...,<NA>,723,<NA>,723,6.725677867752952e+18,2,1017,<NA>,/usr/WS2,corona222
1,read,POSIX,0,2033896,2033896,7096598225222401591,12123992,12124790,798,<NA>,...,<NA>,798,<NA>,798,1.4303593417878847e+19,2,1539,<NA>,/usr/WS2,corona222
2,read,POSIX,0,2033896,2033896,7096598225222401591,12134236,12134955,719,<NA>,...,<NA>,719,<NA>,719,4.536461542053639e+18,2,7596,<NA>,/usr/WS2,corona222
3,read,POSIX,0,2033896,2033896,7096598225222401591,12136550,12137085,535,<NA>,...,<NA>,535,<NA>,535,1.2452020734755443e+19,2,2331,<NA>,/usr/WS2,corona222
4,read,POSIX,0,2033896,2033896,7096598225222401591,12138539,12139932,1393,<NA>,...,<NA>,1393,<NA>,1393,3.78947048789436e+18,2,5431,<NA>,/usr/WS2,corona222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1873,read,POSIX,0,2033896,2033896,7096598225222401591,12094915,12095433,518,<NA>,...,<NA>,518,<NA>,518,1.217967774689493e+18,2,2572,<NA>,/usr/WS2,corona222
1874,read,POSIX,0,2033896,2033896,7096598225222401591,12097077,12099330,2253,<NA>,...,<NA>,2253,<NA>,2253,9.314059654568468e+18,2,8887,<NA>,/usr/WS2,corona222
1875,read,POSIX,0,2033896,2033896,7096598225222401591,12101057,12101293,236,<NA>,...,<NA>,236,<NA>,236,1.798780992960272e+19,2,2636,<NA>,/usr/WS2,corona222
1876,read,POSIX,0,2033896,2033896,7096598225222401591,12110290,12110832,542,<NA>,...,<NA>,542,<NA>,542,1.9146158222955904e+18,2,191,<NA>,/usr/WS2,corona222


In [66]:
new_df.groupby("mount_point").count().compute()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,trange,compute_time,io_time,app_io_time,total_time,fhash,phase,size,value,hostname
mount_point,,,,,,,,,,,,,,,,,,,,
/collab/usr/gapps,1473,1473,1473,1473,1473,1473,1473,1473,1473,0,1473,0,1473,0,1473,1473,1473,1473,0,1473
/g/g92,10,10,10,10,10,10,10,10,10,0,10,0,10,0,10,10,10,10,0,10
/usr/WS2,24808,24808,24808,24808,24808,24808,24808,24808,24808,0,24808,0,24808,0,24808,24808,24808,24808,0,24808
/dev,37,37,37,37,37,37,37,37,37,0,37,0,37,0,37,37,37,37,0,37
/p/lustre3,482,482,482,482,482,482,482,482,482,0,482,0,482,0,482,482,482,482,0,482
/proc,278,278,278,278,278,278,278,278,278,0,278,0,278,0,278,278,278,278,0,278
/sys,1122,1122,1122,1122,1122,1122,1122,1122,1122,0,1122,0,1122,0,1122,1122,1122,1122,0,1122
/usr/share,8,8,8,8,8,8,8,8,8,0,8,0,8,0,8,8,8,8,0,8
/usr/tce,12,12,12,12,12,12,12,12,12,0,12,0,0,0,12,12,12,12,0,12


In [ ]:
app_name = "deepspeed-dlio-step100" 
cp_interference = "/p/lustre3/pandey2/logs/Results_Checkpoint"


# app_name = "deepspeed-dlio-scr-step100"
# burst_data = "/p/lustre3/pandey2/logs/results_checkpoint/"+app_name+"/data_burst.csv"
# df = pd.read_csv(burst_data)
# df.head()


data_events_obj = DFGrepInterferencePartitionBased(app_name=app_name, operation="data", cp_dir = cp_dir, existing=True)
data = data_events_obj.inter.compute()


data["Operation"] = "Data"
data["Interfered"] = np.where(data['deg_caller'] < 1 / data['interference'], "interfered", "not-interfered")
data['valid'] = (data['deg_caller'] > 1) & (data['dur'] > 0) & (data['interference'] > 0)



In [14]:
data.head()

,name,cat,size,size_category,ts,te,dur,trange,mount_point,hostname,deg_caller,deg_other,min_dur,interference,Operation,Interfered,valid
0,read,POSIX,4025,2^1 KiB - 2^2 KiB,64312,65124,812,0,/collab/usr/gapps,corona191,8,1.0,3.0,0.003695,Data,interfered,True
1,read,POSIX,4025,2^1 KiB - 2^2 KiB,64608,65126,518,0,/collab/usr/gapps,corona191,8,1.0,3.0,0.005792,Data,interfered,True
2,read,POSIX,4025,2^1 KiB - 2^2 KiB,64627,65128,501,0,/collab/usr/gapps,corona191,8,1.0,3.0,0.005988,Data,interfered,True
3,read,POSIX,4025,2^1 KiB - 2^2 KiB,64631,65131,500,0,/collab/usr/gapps,corona191,8,1.0,3.0,0.006,Data,interfered,True
4,read,POSIX,4025,2^1 KiB - 2^2 KiB,64922,65130,208,0,/collab/usr/gapps,corona191,8,1.0,3.0,0.014423,Data,interfered,True


In [13]:
# data.groupby(["trange","pid","tid"]).agg({'dur':sum}).groupby("trange").max().sum()

KeyError: 'pid'

# Burst Results

In [ ]:

identifier = "ddf_bur-data"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint"
burst_df = dd.read_parquet(f"{cp_dir}/{app_name}/{identifier}*.parquet")
burst_pandas = burst_df.compute()

# DataFlow

In [6]:
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/montage-pegasus-2mass-2deg-4node/"
df1 = dd.read_parquet(cp_dir+"lvl1/*.parquet")
df2 = dd.read_parquet(cp_dir+"lvl2/*.parquet")
df3 = dd.read_parquet(cp_dir+"lvl3/*.parquet")


fhash_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/fhash/montage-pegasus-2mass-2deg-4node/*.parquet"
fhash = dd.read_parquet(fhash_dir)
fhash["mount_point"] = fhash["mount_point"].where(fhash["mount_point"].str.startswith("/"), "/p/lustre3")

#make fhash end with .0 to make consistant with fhash df values
df1 = df1.assign(fhash=lambda d: d.fhash.astype("string").str.strip().where(d.fhash.astype("string").str.strip().str.endswith(".0", na=False), d.fhash.astype("string").str.strip() + ".0"))
df2 = df2.assign(fhash=lambda d: d.fhash.astype("string").str.strip().where(d.fhash.astype("string").str.strip().str.endswith(".0", na=False), d.fhash.astype("string").str.strip() + ".0"))
df3 = df3.assign(fhash=lambda d: d.fhash.astype("string").str.strip().where(d.fhash.astype("string").str.strip().str.endswith(".0", na=False), d.fhash.astype("string").str.strip() + ".0"))

# Add mount point information to the df3 

df1_with_mount = df1.merge(
    fhash[['hash', 'mount_point']], 
    left_on='fhash', 
    right_on='hash', 
    how='left'
)

df2_with_mount = df2.merge(
    fhash[['hash', 'mount_point']], 
    left_on='fhash', 
    right_on='hash', 
    how='left'
)
df3_with_mount = df3.merge(
    fhash[['hash', 'mount_point']], 
    left_on='fhash', 
    right_on='hash', 
    how='left'
)

shared_mounts = [
    "/p/lustre3",
    "/usr/WS2",
    "/usr/tce",
    "/usr/lib64",
    "/usr",
    "/etc/libibverbs.d",
    "/etc/psm3.conf",
    "/etc/libfabric.conf",
]

node_local_mounts = [
    "/proc",
    "/dev",
    "/sys",
    "/tmp",
    "/dev/shm",
    "/var/tmp",
]

# select only specific mount points for analysis
# df1_new = df1_with_mount[df1_with_mount["mount_point"].isin(node_local_mounts)]
# df2_new = df2_with_mount[df2_with_mount["mount_point"].isin(node_local_mounts)]
# df3_new = df3_with_mount[df3_with_mount["mount_point"].isin(shared_mounts)]

df1_new = df1_with_mount
df2_new = df2_with_mount
df3_new = df3_with_mount

df1_unique_edges = df1_new.drop_duplicates()
df2_unique_edges = df2_new.drop_duplicates()
df3_unique_edges = df3_new.drop_duplicates()


In [73]:
df1_unique_edges.compute()

,hostname,pid,fhash,ts,hash,mount_point
0,corona203,1413027,1001.0,325472934,1001.0,/proc
1,corona203,1409795,10020.0,157518618,10020.0,/proc
2,corona203,1414629,10061.0,483099875,10061.0,/proc
5,corona203,1410851,10096.0,211297775,10096.0,/proc
7,corona203,1406558,10114.0,6075359,10114.0,/proc
...,...,...,...,...,...,...
4544,corona200,2977040,979.0,244099734,979.0,/proc
4545,corona200,2976875,9791.0,227204033,9791.0,/proc
4548,corona200,2977698,9894.0,271752699,9894.0,/proc
4552,corona200,2975321,9940.0,146373040,9940.0,/proc


In [74]:
df2_with_mount.compute()

,pid,fhash,prod,cons,hash,mount_point
0,1407081,56048.0,0,16,56048.0,/p/lustre3
1,1407202,10727.0,0,2,10727.0,/proc
2,1407202,12002.0,0,2,12002.0,/proc
3,1407202,17166.0,0,2,17166.0,/proc
4,1407202,19455.0,0,1,19455.0,/proc
...,...,...,...,...,...,...
420768,2980216,6337.0,0,1,6337.0,/p/lustre3
420769,2980216,64348.0,0,2,64348.0,/proc
420770,2980216,769.0,0,2,769.0,/proc
420771,2980216,8062.0,0,1,8062.0,/proc


In [7]:
from functions import *

In [8]:
# (A) Original: per (fhash, hostname)
l3_results = compute_io_metrics_host(df3_unique_edges, result_df1, compute_result=False)

# (B) New: per (fhash, pid) using df2 (which has columns ['fhash','pid', ...])
l2_results = compute_io_metrics_pid(df2_unique_edges, result_df1, compute_result=False)

l1_results = compute_io_metrics_pid(df1_unique_edges, result_df1, compute_result=False)


In [9]:
l1_results.compute().query('bw > 0').to_csv(
    "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/montage-pegasus-2mass-2deg-4node/all/l1.csv",
    index=False,
)

l2_results.compute().query('bw > 0').to_csv(
    "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/montage-pegasus-2mass-2deg-4node/all/l2.csv",
    index=False,
)

l3_results.compute().query('bw > 0').to_csv(
    "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/montage-pegasus-2mass-2deg-4node/all/l3.csv",
    index=False,
)


In [79]:
out_host.query('prod > 0').compute()

,fhash,hostname,prod,cons,hash,mount_point,io_time,io_count,io_size,iops,bw
4482,55152.0,corona200,1,0,55152.0,/p/lustre3,375496,13913,133467865,0.037052,355.444173
6750,65287.0,corona200,1,0,65287.0,/p/lustre3,498219,16315,156504985,0.032747,314.128897


In [78]:
out_pid.compute()

,hostname,pid,fhash,ts,hash,mount_point,io_time,io_count,io_size,iops,bw
0,corona203,1413027,1001.0,325472934,1001.0,/proc,21751,350,996480,0.016091,45.813066
1,corona203,1409795,10020.0,157518618,10020.0,/proc,12658,16,34560,0.001264,2.730289
2,corona203,1414629,10061.0,483099875,10061.0,/proc,26640,1449,4161600,0.054392,156.216216
3,corona203,1410851,10096.0,211297775,10096.0,/proc,20259,174,489600,0.008589,24.167037
4,corona203,1406558,10114.0,6075359,10114.0,/proc,11693,106,13302,0.009065,1.137604
...,...,...,...,...,...,...,...,...,...,...,...
11704,corona200,2977040,979.0,244099734,979.0,/proc,16663,1171,3360960,0.070275,201.701974
11705,corona200,2976875,9791.0,227204033,9791.0,/proc,20613,77,210240,0.003736,10.199389
11706,corona200,2977698,9894.0,271752699,9894.0,/proc,34681,1444,4147200,0.041637,119.581327
11707,corona200,2975321,9940.0,146373040,9940.0,/proc,22024,226,639360,0.010262,29.030149


In [ ]:
def get_IO_metrics(events):
   io_time = events.groupby("trange","pid","tid").agg({'dur':sum}).groupby("trange").max().sum()
   io_count = events.count()
   io_size = events["size"].sum()
   ipos = io_count/io_time
   bw = io_size/io_time
   return io_time, io_count, io_size, ipos, bw

In [7]:
df3_new.compute()

,fhash,hostname,prod,cons,hash,mount_point
188,35471.0,corona203,0,1,35471.0,/p/lustre3
189,35471.0,corona203,0,1,35471.0,/p/lustre3
190,35471.0,corona203,0,1,35471.0,/p/lustre3
191,35471.0,corona203,0,1,35471.0,/p/lustre3
192,39139.0,corona203,0,1,39139.0,/p/lustre3
...,...,...,...,...,...,...
226778,59458.0,corona203,0,1,59458.0,/p/lustre3
226779,59458.0,corona203,0,1,59458.0,/p/lustre3
226782,59458.0,corona203,0,1,59458.0,/p/lustre3
226784,59458.0,corona203,0,1,59458.0,/p/lustre3


In [44]:
result_df1 = result_df1.query("size > 0")
result_df1.compute()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,compute_time,io_time,app_io_time,total_time,fhash,phase,size,value,mount_point,hostname
1072,open,POSIX,0,2675946,2675946,23424.0,81056537,81056550,13,<NA>,...,<NA>,13,<NA>,13,56559.0,2,5,<NA>,/p/lustre3,corona201
1073,read,POSIX,0,2675946,2675946,23424.0,81056557,81056563,6,<NA>,...,<NA>,6,<NA>,6,56559.0,2,16,<NA>,/p/lustre3,corona201
1079,open,POSIX,0,2675946,2675946,23424.0,81059900,81059905,5,<NA>,...,<NA>,5,<NA>,5,43683.0,2,9,<NA>,./mDiffFit,corona201
1080,read,POSIX,0,2675946,2675946,23424.0,81059911,81059915,4,<NA>,...,<NA>,4,<NA>,4,43683.0,2,16,<NA>,./mDiffFit,corona201
1084,open,POSIX,0,2675946,2676071,23424.0,81060687,81060695,8,<NA>,...,<NA>,8,<NA>,8,13806.0,2,9,<NA>,/dev,corona201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670570,fwrite,STDIO,0,1408901,1408901,36808.0,112313774,112313774,0,<NA>,...,<NA>,<NA>,<NA>,0,10978.0,0,2880,<NA>,/proc,corona203
670571,fwrite,STDIO,0,1408901,1408901,36808.0,112313785,112313814,29,<NA>,...,<NA>,<NA>,<NA>,0,10978.0,0,2880,<NA>,/proc,corona203
670572,fwrite,STDIO,0,1408901,1408901,36808.0,112313820,112313820,0,<NA>,...,<NA>,<NA>,<NA>,0,10978.0,0,2880,<NA>,/proc,corona203
670573,fwrite,STDIO,0,1408901,1408901,36808.0,112313826,112313826,0,<NA>,...,<NA>,<NA>,<NA>,0,10978.0,0,2880,<NA>,/proc,corona203


In [45]:

# df3_new must have: fhash, hostname  (plus anything else you want to keep)
# result_df1 must have: fhash, hostname, trange, pid, tid, dur, size

# --- Normalize dtypes on join keys ---
df3_new = df3_new.assign(
    fhash=df3_new["fhash"].astype("string"),
    hostname=df3_new["hostname"].astype("string"),
)
result_df1 = result_df1.assign(
    fhash=result_df1["fhash"].astype("string"),
    hostname=result_df1["hostname"].astype("string"),
)

# Ensure numeric columns are numeric
for col in ["dur", "size"]:
    if col in result_df1.columns:
        result_df1[col] = dd.to_numeric(result_df1[col], errors="coerce").fillna(0)

# --- Keep only keys that appear in df3_new (reduces shuffle/work) ---
keys = df3_new[["fhash", "hostname"]].drop_duplicates()
filtered = result_df1.merge(keys, on=["fhash", "hostname"], how="inner")

# --- Compute io_time = sum_over_trange( max_{pid,tid} sum(dur) ) ---
# A) sum dur per (fhash, hostname, trange, pid, tid)
agg1 = (
    filtered
    .groupby(["fhash", "hostname", "trange", "pid", "tid"], observed=True)[["dur"]]
    .sum()
    .reset_index()
)

# B) max dur per (fhash, hostname, trange)
agg2 = (
    agg1
    .groupby(["fhash", "hostname", "trange"], observed=True)[["dur"]]
    .max()
    .rename(columns={"dur": "dur_max"})
    .reset_index()
)

# C) sum of dur_max across tranges → io_time
io_time = (
    agg2
    .groupby(["fhash", "hostname"], observed=True)[["dur_max"]]
    .sum()
    .rename(columns={"dur_max": "io_time"})
    .reset_index()
)

# Counts and sizes per (fhash, hostname)
io_count = (
    filtered
    .groupby(["fhash", "hostname"], observed=True)
    .size()
    .to_frame("io_count")
    .reset_index()
)

io_size = (
    filtered
    .groupby(["fhash", "hostname"], observed=True)[["size"]]
    .sum()
    .rename(columns={"size": "io_size"})
    .reset_index()
)

# Combine metrics
metrics = (
    io_time
    .merge(io_count, on=["fhash", "hostname"], how="outer")
    .merge(io_size, on=["fhash", "hostname"], how="outer")
)

# Derived metrics (guard against divide-by-zero)
metrics = metrics.assign(
    iops=(metrics["io_count"] / metrics["io_time"]).where(metrics["io_time"] > 0, 0.0),
    bw=(metrics["io_size"] / metrics["io_time"]).where(metrics["io_time"] > 0, 0.0),
)

# Merge onto df3_new
df3_with_metrics = df3_new.merge(metrics, on=["fhash", "hostname"], how="left")

# Drop rows that had NO matching events (i.e., metrics are NaN)
df3_with_metrics = df3_with_metrics.dropna(subset=["io_time", "io_count", "io_size"])

# (Optional) materialize to Pandas to inspect:
result = df3_with_metrics.compute()
result.head()


,fhash,hostname,prod,cons,hash,mount_point,io_time,io_count,io_size,iops,bw
0,1001.0,corona203,0,1,56048.0,/p/lustre3,5292,346.0,996480,0.065382,188.29932
1,10020.0,corona203,0,1,56048.0,/p/lustre3,747,14.0,34585,0.018742,46.298527
2,10061.0,corona203,0,1,56048.0,/p/lustre3,17424,1447.0,4161625,0.083046,238.84441
3,10062.0,corona203,0,1,56048.0,/p/lustre3,16304,1439.0,4138585,0.088261,253.838629
4,10091.0,corona203,0,1,56048.0,/p/lustre3,2795,80.0,230400,0.028623,82.432916


In [47]:
result

,fhash,hostname,prod,cons,hash,mount_point,io_time,io_count,io_size,iops,bw
0,1001.0,corona203,0,1,56048.0,/p/lustre3,5292,346.0,996480,0.065382,188.29932
1,10020.0,corona203,0,1,56048.0,/p/lustre3,747,14.0,34585,0.018742,46.298527
2,10061.0,corona203,0,1,56048.0,/p/lustre3,17424,1447.0,4161625,0.083046,238.84441
3,10062.0,corona203,0,1,56048.0,/p/lustre3,16304,1439.0,4138585,0.088261,253.838629
4,10091.0,corona203,0,1,56048.0,/p/lustre3,2795,80.0,230400,0.028623,82.432916
...,...,...,...,...,...,...,...,...,...,...,...
741,9851.0,corona200,0,1,53557.0,/p/lustre3,17459,2244.0,6459842,0.12853,370.000687
742,9900.0,corona200,0,1,53557.0,/p/lustre3,9691,548.0,1572484,0.056547,162.262305
743,9929.0,corona200,0,1,53557.0,/p/lustre3,23128,5768.0,16588837,0.249395,717.262063
744,997.0,corona200,0,1,53557.0,/p/lustre3,14291,1435.0,5356804,0.100413,374.83759


In [48]:
result.hash.unique()

<ArrowStringArray>
['56048.0', '36376.0', '39782.0', '49548.0', '46059.0', '43471.0', '60690.0',
 '19619.0', '36051.0', '42916.0', '25451.0', '12229.0', '46435.0', '53943.0',
 '58681.0', '26085.0', '59073.0', '15474.0', '24134.0', '14660.0', '22332.0',
 '15971.0', '14220.0', '19732.0', '20139.0', '64045.0', '39139.0', '30598.0',
 '53557.0']
Length: 29, dtype: string

In [49]:
import dask.dataframe as dd
import pandas as pd

def compute_io_metrics_with_role(
    df3_new: dd.DataFrame,
    result_df1: dd.DataFrame,
    *,
    produce_events = ("write", "pwrite", "fwrite"),
    drop_unmatched: bool = True,
    key_dtype: str = "string",
    compute_result: bool = False,
):
    """
    Compute I/O metrics per (fhash, hostname) for rows in df3_new using events
    from result_df1, with event selection conditioned on per-row role flags:
      - If df3_new.prod == 1  -> include only events whose `name` is in produce_events
      - If df3_new.cons == 1  -> include only events whose `name` is NOT in produce_events
      - If both prod==1 and cons==1 -> include union of both (i.e., all events)
      - If both prod==0 and cons==0 -> include none (row will drop if drop_unmatched=True)

    Required columns:
      df3_new: ['fhash','hostname','prod','cons']
      result_df1: ['fhash','hostname','trange','pid','tid','dur','size','name']

    Returns:
      Dask DataFrame (default) or Pandas DataFrame if compute_result=True.
    """

    # --- Normalize join key dtypes ---
    df3_new = df3_new.assign(
        fhash=df3_new["fhash"].astype(key_dtype),
        hostname=df3_new["hostname"].astype(key_dtype),
        prod=dd.to_numeric(df3_new["prod"], errors="coerce").fillna(0).astype("int8"),
        cons=dd.to_numeric(df3_new["cons"], errors="coerce").fillna(0).astype("int8"),
    )
    result_df1 = result_df1.assign(
        fhash=result_df1["fhash"].astype(key_dtype),
        hostname=result_df1["hostname"].astype(key_dtype),
    )

    # --- Ensure numerics + event name handling ---
    for col in ["dur", "size"]:
        if col in result_df1.columns:
            result_df1[col] = dd.to_numeric(result_df1[col], errors="coerce").fillna(0)
    # name may have nulls; treat null as non-produce by default
    if "name" not in result_df1.columns:
        raise ValueError("result_df1 must contain a 'name' column for event type filtering.")
    result_df1 = result_df1.assign(name=result_df1["name"].astype("string").fillna(""))

    # --- Keep only keys that appear in df3_new ---
    keys = df3_new[["fhash", "hostname", "prod", "cons"]].drop_duplicates()
    merged = result_df1.merge(keys, on=["fhash", "hostname"], how="inner")

    # --- Mark produce vs consume for each event row ---
    # is_produce = True if name in produce_events
    produce_events_set = set(produce_events)
    # Dask-friendly membership test
    is_prod = merged["name"].isin(list(produce_events_set))
    merged = merged.assign(is_produce=is_prod)

    # --- Apply per-row role filtering:
    # keep if (prod==1 & is_produce) OR (cons==1 & ~is_produce)
    keep_mask = (merged["prod"] == 1) & (merged["is_produce"])
    keep_mask |= (merged["cons"] == 1) & (~merged["is_produce"])
    filtered = merged[keep_mask]

    # --- Compute io_time = sum_over_trange( max_{pid,tid} sum(dur) ) ---
    agg1 = (
        filtered
        .groupby(["fhash", "hostname", "trange", "pid", "tid"], observed=True)[["dur"]]
        .sum()
        .reset_index()
    )

    agg2 = (
        agg1
        .groupby(["fhash", "hostname", "trange"], observed=True)[["dur"]]
        .max()
        .rename(columns={"dur": "dur_max"})
        .reset_index()
    )

    io_time = (
        agg2
        .groupby(["fhash", "hostname"], observed=True)[["dur_max"]]
        .sum()
        .rename(columns={"dur_max": "io_time"})
        .reset_index()
    )

    io_count = (
        filtered
        .groupby(["fhash", "hostname"], observed=True)
        .size()
        .to_frame("io_count")
        .reset_index()
    )

    io_size = (
        filtered
        .groupby(["fhash", "hostname"], observed=True)[["size"]]
        .sum()
        .rename(columns={"size": "io_size"})
        .reset_index()
    )

    metrics = (
        io_time
        .merge(io_count, on=["fhash", "hostname"], how="outer")
        .merge(io_size, on=["fhash", "hostname"], how="outer")
    )

    # Derived metrics with /0 guard
    metrics = metrics.assign(
        iops=(metrics["io_count"] / metrics["io_time"]).where(metrics["io_time"] > 0, 0.0),
        bw=(metrics["io_size"] / metrics["io_time"]).where(metrics["io_time"] > 0, 0.0),
    )

    # Merge back to df3_new (only on fhash,hostname so prod/cons stay from df3_new)
    out = df3_new.merge(metrics, on=["fhash", "hostname"], how="left")

    if drop_unmatched:
        out = out.dropna(subset=["io_time", "io_count", "io_size"])

    return out.compute() if compute_result else out


In [50]:
df_final = compute_io_metrics_with_role(
    df3_new,
    result_df1,
    produce_events=("write","pwrite","fwrite"),
    drop_unmatched=True,
    compute_result=False,  # set True to get a Pandas DataFrame
)


In [52]:
df_final.compute()

,fhash,hostname,prod,cons,hash,mount_point,io_time,io_count,io_size,iops,bw
0,1001.0,corona203,0,1,56048.0,/p/lustre3,291,1.0,2880,0.003436,9.896907
1,10020.0,corona203,0,1,56048.0,/p/lustre3,302,2.0,25,0.006623,0.082781
2,10061.0,corona203,0,1,56048.0,/p/lustre3,313,3.0,2905,0.009585,9.28115
3,10062.0,corona203,0,1,56048.0,/p/lustre3,349,3.0,2905,0.008596,8.323782
4,10091.0,corona203,0,1,56048.0,/p/lustre3,363,1.0,2880,0.002755,7.933884
...,...,...,...,...,...,...,...,...,...,...,...
741,9851.0,corona200,0,1,53557.0,/p/lustre3,4921,1123.0,3231362,0.228206,656.647429
742,9900.0,corona200,0,1,53557.0,/p/lustre3,3361,276.0,789124,0.082118,234.788456
743,9929.0,corona200,0,1,53557.0,/p/lustre3,7013,4329.0,12444517,0.617282,1774.492656
744,997.0,corona200,0,1,53557.0,/p/lustre3,10709,1259.0,4849924,0.117565,452.882996
